### Inférence sur les messages français
Ce notebook permet d'inférer le modèle sur l'ensemble des messages français. Il produit un dataset classifié `flat_fench_left_right_interactions`. 

In [ ]:
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from xgboost import XGBClassifier
import re
import unicodedata
import html
import os 

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # 1. HTML entities (&gt etc.)
    text = html.unescape(text)

    # 2. Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # 3. Fix escaped apostrophes (IMPORTANT)
    text = text.replace("\\'", "'")

    # 4. Remove leftover backslashes
    text = text.replace("\\", "")

    # 5. Fix non-breaking spaces
    text = text.replace("\xa0", " ")

    # 6. Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
v_clean = np.vectorize(clean_text)

### Importer les messages à inférer

In [ ]:
df = pd.read_csv('flat_french_political_interactions.csv')
print(len(df))
all_interactions = df['text'].tolist()

In [ ]:
N_CHUNKS = 20
parts = np.array_split(all_interactions, N_CHUNKS)
print(f"Chunk sizes: {[len(p) for p in parts]}")

### Importer les deux parties du modèle

In [ ]:
classifier = XGBClassifier()
classifier.load_model("XGBmodel.json")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-mpnet-base-v2"
)

sbert = SentenceTransformer(MODEL_NAME,
    device=device)

### Inférence

In [ ]:
def predict_with_threshold(model, X, threshold=0.5):

    proba = model.predict_proba(X)[:, 1]

    return (proba >= threshold).astype(int)

In [ ]:
best_treshold = 0.5289310689146067 # reporté manuellement depuis le notebook d'entraînement
for i in range(1, N_CHUNKS + 1):
    chunk_text = parts[i - 1].copy()
    chunk_clean = v_clean(chunk_text)
    X = sbert.encode(
        chunk_clean,
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    y_pred = predict_with_threshold(classifier, X, best_treshold)
    y_labels = ['Left' if l==0 else 'Right' for l in y_pred]

    temp_df = pd.DataFrame(chunk_text, columns=['text'])
    temp_df['left_right'] = y_labels
    temp_df.to_csv(f'output/FR_left_right_{i}.csv', index=False)

### Fusion finale

In [ ]:
list_outputs = os.listdir('output')
first_file = list_outputs.pop(0)

df = pd.read_csv('output/'+first_file)
for file in list_outputs:
    temp = pd.read_csv('output/'+file)
    df = pd.concat([df, temp])
df = df.drop_duplicates()
df.to_csv('flat_french_left_right_interactions.csv', index=False)
print(len(df))